# E04 Breast Cancer：缩放/PCA 与召回权衡

> 对应路线图任务：**E04**（Issue #9）

## 目标
- 数据集：`sklearn breast_cancer`
- 任务类型：`二分类（医疗场景）`
- 用 `wanglaoshi` 完成：环境检查 → EDA → 特征/划分注意点 → 基线 →（如适用）提交生成与校验

## 本案例技术难点
- 特征量纲差异大，需要缩放
- PCA 降维后可解释性下降
- 医疗场景更关注召回率/漏诊成本

## 运行模式
- `QUICK=True`：小样本/少折/少模型，适合课堂演示
- `QUICK=False`：更完整对比（耗时更长）


In [ ]:
# ===== 全局配置 =====
QUICK = True
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

from wanglaoshi import VERSIONS as V
from wanglaoshi import JupyterEnv as JE

JE.no_warning()
V.show_env_info()
V.check_all_versions(version='simple', show_latest=False)


## 1. 数据加载


In [ ]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
df = data.frame
print(df.shape)
df.head()


## 2. 快速 EDA（wanglaoshi.Analyzer）

先用 Analyzer 看缺失、分布、相关性；大数据集请先抽样。


In [ ]:
from wanglaoshi import DataAnalyzer

# 若上面成功得到 df，则继续；否则请先解决数据路径问题
try:
    eda_df = df.copy()
except NameError:
    raise RuntimeError('请先在上一格成功加载为变量 df')

# 过大时抽样，避免 Notebook 卡死
if QUICK and len(eda_df) > 5000:
    eda_df = eda_df.sample(5000, random_state=RANDOM_STATE)

analyzer = DataAnalyzer(eda_df)
print(analyzer.get_basic_info())
display(analyzer.missing_value_analysis().head(20))
display(analyzer.basic_statistics().head(20))


## 3. 训练/验证划分（本案例最容易踩坑的地方）

请根据案例类型选择：
- 分类：`StratifiedShuffleSplit` / `StratifiedKFold`
- 时间：按时间切，禁止随机打乱
- 空间/分组：`GroupKFold`
- 不均衡：关注 PR 曲线，不只看 Accuracy


In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

# ---- 请按数据集修改 target_col / feature_cols ----
# 下面给一个通用探测：优先常见目标列名

candidates = ['target', 'target_name', 'Survived', 'survived', 'Class', 'quality',
              'MedHouseVal', 'Rings', 'cnt', 'income', 'SalePrice', 'label']
target_col = None
for c in candidates:
    if c in df.columns:
        target_col = c
        break

if target_col is None:
    # digits 案例可能把 y 分开了
    if 'y' in globals():
        target_col = None
        print('检测到独立 y，将使用 X/y 变量')
    else:
        raise RuntimeError(f'请手动指定 target_col。当前列: {list(df.columns)}')

if target_col is not None:
    feature_cols = [c for c in df.columns if c != target_col]
    X = df[feature_cols].select_dtypes(include=[np.number]).copy()
    y = df[target_col].copy()
    print('target =', target_col, 'numeric features =', len(X.columns))
else:
    X = X.copy()
    y = y.copy()

# 简单划分（时间/分组案例请在后面覆盖）
stratify = y if (y.dtype == 'object' or y.nunique() < 20) else None
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=stratify
)
print(X_train.shape, X_valid.shape)


## 4. 快速基线（wanglaoshi.CompetitionTools.QuickBaseline）


In [ ]:
from wanglaoshi import CompetitionTools as CT

# 仅数值特征的快速基线；含类别特征的案例请先做编码（见 FeatureEngineering）
baseline = CT.QuickBaseline()

# Quick 模式：只用部分模型会更快——这里仍跑全量小模型，但数据已可能抽样
results = baseline.run_all_models(
    X_train=X_train.fillna(X_train.median(numeric_only=True)),
    y_train=y_train,
    X_test=X_valid.fillna(X_train.median(numeric_only=True)),
    y_test=y_valid,
    task_type='auto',
)
comparison = baseline.compare_models(results, top_n=5)
comparison


## 5. 特征工程尝试（可选增强）

使用 `FeatureEngineering.AdvancedFeatureEngineer`：时间特征、目标编码、变换等。  
**提醒：目标编码 / 缺失填充统计必须只在训练集拟合。**


In [ ]:
from wanglaoshi import FeatureEngineering as FE

engineer = FE.AdvancedFeatureEngineer()
print('FeatureEngineering 已加载。请按本案例难点选择：')
print('- extract_datetime_features')
print('- target_encode / frequency_encode / one_hot_encode')
print('- log_transform / bin_features')
print('- select_features_by_importance')


## 6. 提交文件生成与校验（若为竞赛格式）

对 Iris / Titanic / Digits 等可本地模拟 sample_submission。


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from wanglaoshi import CompetitionTools as CT

# 用验证集预测模拟“测试提交”
best_name = comparison.iloc[0]['Model']
best_model = results[best_name]['model']
pred = results[best_name].get('predictions')
if pred is None:
    pred = best_model.predict(X_valid.fillna(X_train.median(numeric_only=True)))

out_dir = Path('outputs') / '04_breast_cancer'
out_dir.mkdir(parents=True, exist_ok=True)

# 构造样本提交：兼容数值标签与字符串标签
sample = pd.DataFrame({
    'Id': np.arange(len(pred)),
    'Target': y_valid.to_numpy()[:len(pred)] if hasattr(y_valid, 'to_numpy') else list(y_valid)[:len(pred)]
})
sample_path = out_dir / 'sample_submission.csv'
sub_path = out_dir / 'submission.csv'
sample.to_csv(sample_path, index=False)

gen = CT.SubmissionGenerator()
sub = gen.create_submission(
    predictions=pred,
    sample_submission_path=str(sample_path),
    output_path=str(sub_path),
    prediction_type='label',
)
val = gen.validate_submission(str(sub_path), str(sample_path))
print(val)
sub.head()


## 7. 误差 / 错分分析与局限

请记录：
1. 哪个模型最好？为什么？
2. 主要错误来自哪一类样本？
3. 本 Notebook 的简化假设（抽样、只用数值特征等）会如何影响结论？
4. 若上线/提交，还缺哪些验证（时间切分、分组公平性、阈值调优）？


In [ ]:
# 简单错误分析模板（分类时）
from sklearn.metrics import classification_report, confusion_matrix

if str(comparison.columns[1]).lower().find('accuracy') >= 0 or 'Test Accuracy' in comparison.columns:
    y_hat = results[best_name]['predictions']
    print(classification_report(y_valid, y_hat))
    print('confusion matrix:\n', confusion_matrix(y_valid, y_hat))
else:
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    y_hat = results[best_name]['predictions']
    print('MAE', mean_absolute_error(y_valid, y_hat))
    print('RMSE', mean_squared_error(y_valid, y_hat, squared=False))


## 8. 检查清单（E04）

- [ ] 数据加载成功，并记录来源/许可
- [ ] EDA 看过缺失与目标分布
- [ ] 划分方式符合任务（随机/分层/时间/分组）
- [ ] 基线对比已跑通
- [ ] 提交生成与校验通过（如适用）
- [ ] 写下至少 2 条可改进方向

---
下一步：打开下一个 Notebook，或回到 `examples/advanced/README.md` 查看学习顺序。
